# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset—ordered logistic regression outputs for adoption predictors of indigenous and modern knowledge in rangeland management in Northern Kenya—using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata overview
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Temporal coverage: {metadata.temporalCoverage}")
print(f"Spatial coverage: {metadata.spatialCoverage}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

All entities will be referenced by their `@id` fields for consistent access. Here we enumerate record sets, their fields, and any available columns.

In [ ]:
# Enumerate available record sets, fields, and columns by their '@id' values
record_sets = list(dataset.record_sets())  # Returns mlcroissant.RecordSet objects
print(f"Record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} -- name: {field.name} (dataType: {field.dataType})")
        if field.columns:
            print(f"      Columns:")
            for col in field.columns:
                print(f"        Column @id: {col.id} -- name: {col.name}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id` values from the overview above. Data will be dynamically referenced using these identifiers.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet {record_set_id} with shape: {dataframes[record_set_id].shape}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}\n")
    else:
        print(f"No records found for RecordSet {record_set_id}\n")

# Display the head of the first available dataframe
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"Displaying first few rows for RecordSet @id: {chosen_record_set_id}")
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field for analysis and demonstrate filtering, normalization, and grouping using that field and another categorical/grouping field. All fields are referenced by their `@id` where possible.

In [ ]:
# Select a record set and fields for EDA
if dataframes:
    df = dataframes[chosen_record_set_id]
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    print(f"Numeric fields in {chosen_record_set_id}: {numeric_fields}")
    
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Using the first numeric field as example
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a categorical field (for demonstration, choose string columns except numeric_field_id)
        group_fields = [col for col in df.select_dtypes(include='object').columns.tolist() if col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}, showing mean {numeric_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll produce a histogram of the selected numeric field for the filtered records, and a bar chart of group means if grouping was done.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if EDA produced filtered_df and numeric_field_id
if 'filtered_df' in locals() and 'numeric_field_id' in locals():
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} (filtered records)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Visualization by grouping
    if 'grouped_df' in locals() and 'group_field_id' in locals():
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
This notebook illustrated exploration of a Croissant-structured dataset using `mlcroissant`. Using `@id` references for record sets and fields enabled robust, reproducible access to different data facets.

Key findings observed include basic data distributions, filtered analyses on numeric predictors, and aggregation by categorical attributes. To fully understand the dataset structure for advanced analysis, consult the FAIR² schema metadata and documentation. Further exploration can include model fit diagnostics, deeper statistical summaries, and visual analysis tailored to the dataset's fields and usage context.